# Analisis Perbandingan Gabungan: B0 vs B1 vs B2 (WarisAI)

Notebook ini membandingkan secara langsung performa, efisiensi, dan pola error dari tiga konfigurasi baseline pada **split `dev.jsonl` (30 query)** dengan **base generator identik (`Qwen/Qwen2.5-1.5B-Instruct`, `seed=42`)**:
- **B0:** Direct LLM (zero-shot, tanpa retrieval)
- **B1:** Structured Prompted LLM (JSON schema + few-shot + kebijakan abstain, tanpa retrieval)
- **B2:** Naive RAG (Dense FAISS `multilingual-e5-small` + `fixed` chunk + `top_k=4`)

In [ ]:
import json, statistics
from pathlib import Path
from collections import Counter

ROOT = Path('..') if (Path('..') / 'runs').exists() else Path('.')
CFGS = ['B0', 'B1', 'B2']

summaries = {c: json.loads((ROOT / 'runs' / c / 'summary.json').read_text(encoding='utf-8')) for c in CFGS}
outputs = {c: [json.loads(l) for l in (ROOT / 'runs' / c / 'dev_outputs.jsonl').open(encoding='utf-8') if l.strip()] for c in CFGS}
by_qid = {c: {r['query_id']: r for r in rows} for c, rows in outputs.items()}

print('Loaded configs:', {c: len(v) for c, v in outputs.items()})

In [ ]:
# 1. Tabel Perbandingan Utama B0 vs B1 vs B2 (Kualitas + Efisiensi + Delta)
def extra_stats(rows):
    lats = sorted(r['latency_s'] for r in rows)
    p95 = lats[min(len(lats) - 1, int(0.95 * len(lats)))]
    return {
        'avg_in_tok': round(statistics.mean(r['in_tokens'] for r in rows), 1),
        'avg_out_tok': round(statistics.mean(r['out_tokens'] for r in rows), 1),
        'trunc_rate': round(sum(1 for r in rows if r['out_tokens'] >= 319) / len(rows), 3),
        'p95_latency_s': round(p95, 2),
    }

full_stats = {c: {**summaries[c], **extra_stats(outputs[c])} for c in CFGS}
metrics = [
    'json_valid', 'abstain_acc_unans', 'false_refusal_ans', 'citation_rate_ans',
    'trunc_rate', 'avg_in_tok', 'avg_out_tok', 'avg_latency_s', 'p95_latency_s'
]

print(f"{'Metrik':20s} | {'B0':>8s} | {'B1':>8s} | {'B2':>8s} | {'B1 - B0':>9s} | {'B2 - B1':>9s}")
print('-' * 76)
for m in metrics:
    v0, v1, v2 = full_stats['B0'][m], full_stats['B1'][m], full_stats['B2'][m]
    print(f'{m:20s} | {v0:8.3f} | {v1:8.3f} | {v2:8.3f} | {v1-v0:+9.3f} | {v2-v1:+9.3f}')

In [ ]:
# 2. Perbandingan Per Kategori Query (JSON Valid, Verdict Supported/Unknown, Latensi)
cats = ['pasal_definisi', 'multi_hop', 'analisis_putusan', 'konflik_konteks', 'unanswerable']
print(f"{'Kategori':17s} | {'B0 (json/lat)':14s} | {'B1 (json/sup/unk/lat)':22s} | {'B2 (json/sup/unk/lat)':22s}")
print('-' * 85)
for cat in cats:
    cells = []
    for c in CFGS:
        items = [r for r in outputs[c] if r['category'] == cat]
        jv = sum(1 for x in items if x['parsed_json'] is not None)
        sup = sum(1 for x in items if (x['parsed_json'] or {}).get('verdict') == 'supported')
        unk = sum(1 for x in items if (x['parsed_json'] or {}).get('verdict') == 'unknown')
        lat = statistics.mean(x['latency_s'] for x in items)
        if c == 'B0':
            cells.append(f'{jv}/{len(items)} ({lat:5.2f}s)')
        else:
            cells.append(f'{jv}/{len(items)} s={sup} u={unk} ({lat:5.2f}s)')
    print(f'{cat:17s} | {cells[0]:14s} | {cells[1]:22s} | {cells[2]:22s}')

In [ ]:
# 3. Visualisasi Perbandingan 3 Panel (Kualitas, Latensi/Token, Sumber Retrieval B2)
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # Panel 1: Metrik Kualitas
    qm = ['json_valid', 'abstain_acc_unans', 'false_refusal_ans', 'citation_rate_ans']
    x = range(len(qm)); w = 0.25
    axes[0].bar([i - w for i in x], [full_stats['B0'][m] for m in qm], width=w, label='B0', color='#9ecae1')
    axes[0].bar(x, [full_stats['B1'][m] for m in qm], width=w, label='B1', color='#4292c6')
    axes[0].bar([i + w for i in x], [full_stats['B2'][m] for m in qm], width=w, label='B2', color='#08519c')
    axes[0].set_xticks(list(x)); axes[0].set_xticklabels(['JSON\nValid', 'Abstain\n(Unans)', 'False\nRefusal', 'Citation\nRate'])
    axes[0].set_ylim(0, 1.08); axes[0].set_title('Metrik Kualitas (Dev n=30)'); axes[0].legend()

    # Panel 2: Latensi Mean & P95
    x2 = range(3); w2 = 0.35
    axes[1].bar([i - w2/2 for i in x2], [full_stats[c]['avg_latency_s'] for c in CFGS], width=w2, label='Mean', color='#fdae6b')
    axes[1].bar([i + w2/2 for i in x2], [full_stats[c]['p95_latency_s'] for c in CFGS], width=w2, label='P95', color='#e6550d')
    axes[1].set_xticks(list(x2)); axes[1].set_xticklabels(CFGS)
    axes[1].set_ylabel('Detik'); axes[1].set_title('Latensi Mean vs P95'); axes[1].legend()

    # Panel 3: Komposisi Dokumen Top-4 di B2 (120 slot)
    src_cnt = Counter(d.split('_')[0] for r in outputs['B2'] for d in r.get('retrieved_doc_ids', []))
    labels = ['ART (Badilag)', 'PA (Putusan)', 'KHI (Norma)']
    vals = [src_cnt.get('ART', 0), src_cnt.get('PA', 0), src_cnt.get('KHI', 0)]
    axes[2].bar(labels, vals, color=['#74c476', '#31a354', '#de2d26'])
    for idx, v in enumerate(vals):
        axes[2].text(idx, v + 1.5, f'{v} ({v/120*100:.1f}%)', ha='center', fontweight='bold')
    axes[2].set_ylim(0, max(vals) + 12); axes[2].set_title('Distribusi Sumber Top-4 B2 (120 slot)')

    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib belum terpasang; lewati grafik.')

In [ ]:
# 4. Studi Kasus Side-by-Side (Query Sama Dijawab oleh B0 vs B1 vs B2)
CASE_STUDIES = [
    ('Q_016', 'Pasal 182 KHI — membuktikan efek retrieval vs halusinasi B0/B1'),
    ('Q_056', 'Analisis Putusan Akta Hibah — B2 mulai grounded ke PA_0219/PA_0240'),
    ('Q_082', 'Unanswerable (Adat Batak Toba) — B1 abstain benar, B2 terdistraksi konteks!'),
    ('Q_081', 'Unanswerable (KUHPerdata) — B2 kontaminasi lintas rezim ke Wasiat Wajibah KHI'),
]

for qid, note in CASE_STUDIES:
    print('=' * 90)
    print(f'{qid} | {note}')
    print('QUERY:', by_qid['B0'][qid]['query'])
    for c in CFGS:
        r = by_qid[c][qid]
        pj = r['parsed_json']
        v = pj.get('verdict') if pj else 'RAW/PARSE_FAIL'
        ans = (pj.get('answer') if pj else r['raw_output']) or ''
        extra = f" ret={r['retrieved_doc_ids']}" if c == 'B2' else ''
        print(f'  [{c}] verdict={v:14s} tok={r["out_tokens"]:3d} lat={r["latency_s"]:5.2f}s{extra}')
        print('       ', str(ans)[:260].replace('\n', ' '))

In [ ]:
# 5. Kaitan dengan Hasil Ablation Retrieval A1 & A2 (runs/retrieval_ablation_a1_a2.json)
a1a2_path = ROOT / 'runs' / 'retrieval_ablation_a1_a2.json'
if a1a2_path.exists():
    rep = json.loads(a1a2_path.read_text(encoding='utf-8'))
    print(f"=== RETRIEVAL ABLATION A1 & A2 (qrels: {rep['qrels_used']}) ===")
    print(f"{'Chunking':13s} | {'Retriever':10s} | {'R@5':>6s} | {'R@10':>6s} | {'MRR@10':>6s} | {'nDCG@10':>7s}")
    print('-' * 62)
    for strat, modes in rep['results'].items():
        for mode, m in modes.items():
            print(f"{strat:13s} | {mode:10s} | {m['Recall@5']:6.4f} | {m['Recall@10']:6.4f} | {m['MRR@10']:6.4f} | {m['nDCG@10']:7.4f}")

### Sintesis Perbandingan B0 → B1 → B2 & Peta Jalan ke B3–B4

| Transisi | Apa yang Membaik | Apa yang Masih Gagal / Muncul Baru | Solusi di Tahap Berikutnya |
|---|---|---|---|
| **B0 → B1** *(+Prompt terstruktur)* | `json_valid` **0% → 90%**, latensi turun **14,34s → 8,23s**, `abstain` mulai aktif (**33,3%**), truncasi **16/30 → 0/30** | *Confident hallucination* di balik JSON rapi (contoh `Q_016`: mengarang Pasal 182 KUHPerdata 3/5) | Tambah retrieval (**B2**) |
| **B1 → B2** *(+Naive RAG Dense/Fixed)* | `false_refusal` turun **12,5% → 4,2%**, mulai grounded pada KHI (`Q_016`) & putusan nyata (`Q_056`) | (1) `KHI_*` hanya muncul **1/120 slot (0,8%)** karena Dense+Fixed lemah pada pasal; (2) *Context distraction* pada `unanswerable` (`Q_082`, `Q_081`); (3) 5/30 JSON terpotong di 320 token | **B3:** Hybrid BM25 + Dense + Heading + Reranker + filter `doc_type`, `max_new_tokens=512`<br>**B4:** Evidence Sufficiency Check + Claim Verifier + Abstain |